In [3]:
import pandas as pd
import numpy as np
import json
import datetime

notifications_file = "data/tbl_notifications.csv"
workorders_file = "data/tbl_workorders.csv"

df_notifications = pd.read_csv(notifications_file)
df_workorders = pd.read_csv(workorders_file)

# Notification No and Workorder No
df_notifications['workorder_no'] = pd.to_numeric(df_notifications['workorder_no'], errors='coerce')
df_notifications['notification_no'] = pd.to_numeric(df_notifications['notification_no'], errors='coerce')

df_workorders['workorder_no'] = pd.to_numeric(df_workorders['workorder_no'], errors='coerce')

df_notifications = df_notifications[['workorder_no', 'notification_no', 'notification_type', 
                                     'work_request', 'functional_location','system_status',
                                     'maint_work_centre', 'reported_by', 'notification_date', 'staff_details'
                                     ]]
df_workorders = df_workorders[['workorder_no', 'workorder_type', 'functional_location',
                               'status', 'operations', 'work_centre', 'date_opened', 'labour_details'
                               ]]

df_all_workorders = pd.DataFrame({
    'workorder_no': pd.concat([
        df_workorders['workorder_no'],
        df_notifications['workorder_no']
    ])
}).dropna().drop_duplicates()

def extract_operation_fields(value):
    try:
        data = json.loads(value)
        if isinstance(data, list) and len(data) > 0:
            first = data[0]
            return pd.Series({
                'oper_activity': str(first.get('operation_no', '')).zfill(4),
                'oper_shorttext': first.get('operation_text', '')
            })
    except:
        pass
    
    return pd.Series({
        'oper_activity': '',
        'oper_shorttext': ''
    })

def extract_workorder_dates(value):
    try:
        data = json.loads(value)
        if isinstance(data, list) and len(data) > 0:
            first = data[0]
            start_dt = first.get('plan_start_date_time', None)
            end_dt = first.get('plan_end_date_time', None)
            return pd.Series({'plan_start': start_dt, 'plan_end': end_dt})
    except:
        pass
    return pd.Series({'plan_start': None, 'plan_end': None})

def extract_notification_dates(value):
    try:
        data = json.loads(value)
        if isinstance(data, dict):
            start_dt = data.get('plan_start_date_time', None)
            end_dt = data.get('plan_end_date_time', None)
            return pd.Series({'plan_start': start_dt, 'plan_end': end_dt})
    except:
        pass
    return pd.Series({'plan_start': None, 'plan_end': None})

# Combining all columns into a single dataframe
df_result = df_all_workorders.merge(
    df_workorders,
    on='workorder_no',
    how='left'
)

df_result = df_result.merge(
    df_notifications,
    on='workorder_no',
    how='left'
)

df_result['workorder_no'] = df_result['workorder_no'].astype('Int64')
df_result['notification_no'] = df_result['notification_no'].astype('Int64')
df_result['maintenance_type'] = np.where(
    (df_result['notification_type'] == 'M7') | 
    (df_result['workorder_type'] == 'RAIL - Preventive Maintenance Order'),
    'YPM1',
    'YCM1'
)
df_result['maint_plan'] = "0805"
df_result['functional_location'] = df_result['functional_location_y'].combine_first(df_result['functional_location_x'])
df_result[['oper_activity', 'oper_shorttext']] = df_result['operations'].apply(extract_operation_fields)
df_result['work_centre'] = df_result['maint_work_centre'].combine_first(df_result['work_centre'])
df_result['loc_code'] = df_result['work_centre'].map({
    "RSDM": "RSD",
    "TNM1": "TNM",
    "WES": "WES"
})
df_result['created_on'] = pd.to_datetime(
    df_result['notification_date'].combine_first(df_result['date_opened']),
    errors='coerce'
).dt.strftime('%Y-%m-%d %H:%M:%S')
df_result['dt_created'] = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
df_result['quantity'] = 0.00

df_result[['plan_start_wo', 'plan_end_wo']] = df_result['labour_details'].apply(extract_workorder_dates)
df_result[['plan_start_notif', 'plan_end_notif']] = df_result['staff_details'].apply(extract_notification_dates)

df_result['plan_start'] = df_result['plan_start_notif'].combine_first(df_result['plan_start_wo'])
df_result['plan_start'] = pd.to_datetime(df_result['plan_start'], errors='coerce')

df_result['plan_end'] = df_result['plan_end_notif'].combine_first(df_result['plan_end_wo'])
df_result['plan_end'] = pd.to_datetime(df_result['plan_end'], errors='coerce')

# Finalize column names and drop unnecessary columns
df_result.rename(columns={
    'status': 'user_status',
    'reported_by': 'created_by'
}, inplace=True)

cols_to_drop = [
    'notification_type',
    'workorder_type',
    'functional_location_x',
    'functional_location_y',
    'operations',
    'maint_work_centre',
    'notification_date',
    'date_opened',
    'plan_start_wo','plan_end_wo','plan_start_notif','plan_end_notif',
    'staff_details',
    'labour_details'
]

df_result = df_result.drop(columns=cols_to_drop)
df_result = df_result.sort_values(by='workorder_no', ascending=True)
df_result.head(5)

,workorder_no,user_status,work_centre,notification_no,work_request,system_status,created_by,maintenance_type,maint_plan,functional_location,oper_activity,oper_shorttext,loc_code,created_on,dt_created,quantity,plan_start,plan_end
5310,4000410420,NaN,RSDM,11808988,Preventive Maintenance for RSV021(WEK1),NOPR NOPT ORAS,10011808,YPM1,0805,RSV021 Revenue Service Vehicle 21,,,RSD,NaN,2026-04-09 13:31:44,0.0,2021-10-07 22:15:00,2021-11-07 03:10:00
4110,4000436996,3PLN,TNM1,<NA>,NaN,NaN,NaN,YPM1,0805,PRAL-MRL-TN-DPT Brickfields Depot,0010,MONTHLY MECH. INSP. BOGIE DROP PIT 1,TNM,2022-01-24 00:00:00,2026-04-09 13:31:44,0.0,NaT,NaT
996,4000436997,3PLN,TNM1,<NA>,NaN,NaN,NaN,YPM1,0805,PRAL-MRL-TN-DPT Brickfields Depot,0010,MONTHLY INSPECTION OVERHEAD CRANE 1,TNM,2022-01-06 00:00:00,2026-04-09 13:31:44,0.0,NaT,NaT
736,4000436998,3PLN,TNM1,<NA>,NaN,NaN,NaN,YPM1,0805,PRAL-MRL-TN-DPT Brickfields Depot,0010,MONTHLY INSPECTION HYDRAULIC PRESS,TNM,2022-01-05 00:00:00,2026-04-09 13:31:44,0.0,NaT,NaT
737,4000437236,3PLN,TNM1,<NA>,NaN,NaN,NaN,YPM1,0805,PRAL-MRL-TN-SWT Switch,0010,MONTHLY LUBRICATION TRAVERSER,TNM,2022-01-05 00:00:00,2026-04-09 13:31:44,0.0,NaT,NaT


In [4]:
output_path = 'data/mo_migrations.csv'

df_result.to_csv(output_path, index=False)

print(f"✅ Exported successfully to '{output_path}' (replaced existing sheet)")

✅ Exported successfully to 'data/mo_migrations.csv' (replaced existing sheet)
